# Task 7: vLLM vs. llama.cpp on the same GPU

Runs Qwen3-4B at 16-bit precision with **vLLM** and then with **llama.cpp's llama-server** on the same Colab GPU, and puts both through the same speed test (`engine_bench.py`): 1, 2, 4, 8, 16 and 32 users at once, about 500 prompt tokens and exactly 256 generated tokens per request.

**Before you start:** Runtime > Change runtime type > **T4 GPU**. Then run the cells one at a time, top to bottom. If a cell prints a red error, copy it to Claude.

In [ ]:
# 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# 2. Install vLLM (a few minutes)
!pip install -q vllm
!python -c "import vllm, torch; print('vLLM', vllm.__version__, '| torch', torch.__version__)"

In [ ]:
# 3. Download Qwen3-4B (about 8 GB)
from huggingface_hub import snapshot_download
snapshot_download("Qwen/Qwen3-4B", local_dir="/content/Qwen3-4B")
print("done")

In [ ]:
%%writefile /content/engine_bench.py
"""Speed test for any server with an OpenAI-style /v1/completions endpoint.

Works the same against llama-server and vLLM, so both engines get exactly the
same test. For each number of users it sends that many requests at once, with
streaming on, and records:
  - time to first token (how long a user waits before text starts appearing)
  - decode speed per user (tokens per second once text is flowing)
  - total throughput (all users' tokens / wall time of the round)

Every prompt starts with a different request number, so neither engine can
reuse a cached copy of an earlier prompt.

Example:
    python engine_bench.py --url http://localhost:8000 --engine vllm --hardware "Colab T4"
"""
import argparse
import csv
import json
import os
import threading
import time
import urllib.request

ap = argparse.ArgumentParser()
ap.add_argument("--url", default="http://127.0.0.1:8080")
ap.add_argument("--engine", required=True, help="label for the CSV, e.g. vllm or llama.cpp")
ap.add_argument("--hardware", required=True, help="label for the CSV, e.g. 'Colab T4'")
ap.add_argument("--model", default="", help="model name to send (vLLM needs it; llama-server ignores it)")
ap.add_argument("--users", default="1,2,4,8,16,32", help="comma-separated concurrency levels")
ap.add_argument("--rounds", type=int, default=3, help="rounds per concurrency level")
ap.add_argument("--prompt-words", type=int, default=380, help="about 500 tokens")
ap.add_argument("--max-tokens", type=int, default=256)
ap.add_argument("--out", default="results/runs.csv")
args = ap.parse_args()

DIRECT = urllib.request.build_opener(urllib.request.ProxyHandler({}))
FIELDS = ["engine", "hardware", "users", "round", "request", "prompt_tokens", "tokens",
          "ttft_s", "decode_tok_s", "request_s", "round_wall_s"]

WORDS = ("the model reads every word of this passage before it writes anything and the time it "
         "takes to do that is the prefill while the time it then spends writing one token after "
         "another is the decode ").split()
BODY = " ".join(WORDS[i % len(WORDS)] for i in range(args.prompt_words))


def one_request(n, out):
    payload = {
        "prompt": f"Request {n}. {BODY}\nNow continue the story:",
        "max_tokens": args.max_tokens,
        "temperature": 0,
        "ignore_eos": True,          # always generate exactly max_tokens
        "stream": True,
        "stream_options": {"include_usage": True},
    }
    if args.model:
        payload["model"] = args.model
    req = urllib.request.Request(args.url.rstrip("/") + "/v1/completions",
                                 data=json.dumps(payload).encode(),
                                 headers={"Content-Type": "application/json"})
    t0 = time.perf_counter()
    first = None
    chunks = 0
    usage = {}
    with DIRECT.open(req, timeout=600) as resp:
        for raw in resp:
            line = raw.decode("utf-8", "replace").strip()
            if not line.startswith("data:"):
                continue
            data = line[5:].strip()
            if data == "[DONE]":
                break
            msg = json.loads(data)
            if msg.get("usage"):
                usage = msg["usage"]
            choices = msg.get("choices") or []
            if choices and choices[0].get("text"):
                chunks += 1
                if first is None:
                    first = time.perf_counter()
    end = time.perf_counter()
    tokens = usage.get("completion_tokens") or chunks
    decode = (tokens - 1) / (end - first) if first and end > first and tokens > 1 else None
    out.update(prompt_tokens=usage.get("prompt_tokens", ""), tokens=tokens,
               ttft_s=round(first - t0, 4) if first else "",
               decode_tok_s=round(decode, 2) if decode else "",
               request_s=round(end - t0, 4))


def run_round(users, rnd, counter):
    results = [dict() for _ in range(users)]
    errors = []
    barrier = threading.Barrier(users)

    def worker(i):
        barrier.wait()
        try:
            one_request(counter + i, results[i])
        except Exception as e:  # keep going; report at the end of the round
            errors.append(repr(e))

    threads = [threading.Thread(target=worker, args=(i,)) for i in range(users)]
    t0 = time.perf_counter()
    for t in threads:
        t.start()
    for t in threads:
        t.join()
    wall = time.perf_counter() - t0
    if errors:
        raise SystemExit(f"{len(errors)} request(s) failed, first error: {errors[0]}")
    rows = []
    for i, r in enumerate(results):
        rows.append(dict(engine=args.engine, hardware=args.hardware, users=users, round=rnd,
                         request=i, round_wall_s=round(wall, 4), **r))
    return rows, wall


# Check the server is up before starting.
try:
    DIRECT.open(args.url.rstrip("/") + "/health", timeout=10).read()
except Exception as e:
    raise SystemExit(f"Can't reach the server at {args.url}: {e}\nIs it running and finished loading?")

os.makedirs(os.path.dirname(args.out) or ".", exist_ok=True)
new_file = not os.path.exists(args.out)
counter = 0
print("Warm-up request ...")
run_round(1, 0, 10_000)

print(f"{'users':>5} {'TTFT p50':>9} {'per-user tok/s':>15} {'total tok/s':>12}")
with open(args.out, "a", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=FIELDS)
    if new_file:
        w.writeheader()
    for users in [int(u) for u in args.users.split(",")]:
        ttfts, decs, thr = [], [], []
        for rnd in range(1, args.rounds + 1):
            rows, wall = run_round(users, rnd, counter)
            counter += users
            w.writerows(rows)
            f.flush()
            ttfts += [r["ttft_s"] for r in rows if r["ttft_s"] != ""]
            decs += [r["decode_tok_s"] for r in rows if r["decode_tok_s"] != ""]
            thr.append(sum(r["tokens"] for r in rows) / wall)
        ttfts.sort()
        print(f"{users:>5} {ttfts[len(ttfts) // 2]:>8.3f}s {sum(decs) / len(decs):>15.1f} "
              f"{sum(thr) / len(thr):>12.1f}")
print(f"Saved {args.out}")

In [ ]:
# 5. Helpers to start a server in the background, wait until it is ready, and stop it
import os, signal, subprocess, time, urllib.request
os.makedirs("/content/results", exist_ok=True)
DIRECT = urllib.request.build_opener(urllib.request.ProxyHandler({}))

def start_server(cmd, log_path, port, timeout=1200):
    log = open(log_path, "w")
    p = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)
    t0 = time.time()
    while time.time() - t0 < timeout:
        if p.poll() is not None:
            print(open(log_path).read()[-3000:])
            raise SystemExit("The server stopped while loading. The end of its log is above.")
        try:
            if DIRECT.open(f"http://127.0.0.1:{port}/health", timeout=5).status == 200:
                print(f"Ready after {time.time() - t0:.0f} s")
                return p
        except Exception:
            pass
        time.sleep(5)
    stop_server(p)
    raise SystemExit("The server didn't become ready in time.")

def stop_server(p):
    try:
        os.killpg(p.pid, signal.SIGTERM)
        p.wait(timeout=60)
    except Exception:
        os.killpg(p.pid, signal.SIGKILL)
    time.sleep(5)
    print("Stopped")

## vLLM

In [ ]:
# 6. Start vLLM. The T4 can't do bfloat16, so both engines use float16.
vllm_cmd = ["vllm", "serve", "/content/Qwen3-4B", "--served-model-name", "qwen3-4b",
            "--dtype", "half", "--max-model-len", "2048",
            "--gpu-memory-utilization", "0.90", "--port", "8000"]
p = start_server(vllm_cmd, "/content/results/server_vllm.log", 8000)
!grep -iE "KV cache|concurrency|attention backend|cuda graph" /content/results/server_vllm.log | head -20

In [ ]:
# 7. Speed test vLLM, then stop it to free the GPU
!cd /content && python engine_bench.py --url http://127.0.0.1:8000 --model qwen3-4b --engine vLLM --hardware "Colab T4"
stop_server(p)

## llama.cpp

In [ ]:
# 8. Build llama.cpp b11630 (the same version as on your PC) with CUDA for the T4.
#    This compiles a lot of GPU code: expect 15 to 25 minutes.
!git clone -q --depth 1 --branch b11630 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!cd /content/llama.cpp && cmake -B build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DLLAMA_CURL=OFF > /dev/null && cmake --build build -j 4 --target llama-server llama-bench 2>&1 | tail -3
!ls -la /content/llama.cpp/build/bin/

In [ ]:
# 9. Convert the same Hugging Face files to a 16-bit GGUF (a few minutes)
!pip install -q /content/llama.cpp/gguf-py
!python /content/llama.cpp/convert_hf_to_gguf.py /content/Qwen3-4B --outtype f16 --outfile /content/Qwen3-4B-F16.gguf 2>&1 | tail -2
!ls -la /content/Qwen3-4B-F16.gguf

In [ ]:
# 10. Start llama-server with 32 slots (32 users at once), 1,024 tokens each, then speed test it.
#     Port 8081, because Colab itself already uses 8080.
llama_cmd = ["/content/llama.cpp/build/bin/llama-server", "-m", "/content/Qwen3-4B-F16.gguf",
             "-ngl", "99", "-c", "32768", "-np", "32", "--cache-ram", "0", "--port", "8081"]
p = start_server(llama_cmd, "/content/results/server_llamacpp.log", 8081)
!cd /content && python engine_bench.py --url http://127.0.0.1:8081 --engine llama.cpp --hardware "Colab T4"
stop_server(p)

In [ ]:
# 11. One-user reference numbers from llama-bench (prompt reading and generation speed)
!/content/llama.cpp/build/bin/llama-bench -m /content/Qwen3-4B-F16.gguf -ngl 99 -p 512 -n 256 -r 3 | tee /content/results/llama_bench_t4.txt

In [ ]:
# 12. Download the results to your PC
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv > /content/results/gpu.txt
!python -c "import vllm, torch; print('vLLM', vllm.__version__, '| torch', torch.__version__)" > /content/results/versions.txt
!cd /content && zip -qr results_colab.zip results
from google.colab import files
files.download("/content/results_colab.zip")

In [ ]:
# 13. Upload the 16-bit GGUF to Hugging Face so the PC can download it (the PC's 8 GB of RAM is too little to convert it)
from huggingface_hub import login, HfApi
login()  # paste a Hugging Face token with Write access when the box appears
api = HfApi()
api.create_repo("bbrihuang/Qwen3-4B-GGUF", exist_ok=True)
api.upload_file(path_or_fileobj="/content/Qwen3-4B-F16.gguf", path_in_repo="Qwen3-4B-F16.gguf",
                repo_id="bbrihuang/Qwen3-4B-GGUF")
print("uploaded")